# Import Libraries

In [1]:
import sys
import os
from sklearn.metrics import precision_score, recall_score, f1_score
import json, re
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
this_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(this_dir, os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

In [5]:
from kg_rag.util import *


In [6]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Entity Extraction

In [7]:
DRUG_ENTITY_EXTRACTION= """ You are an expert drug entity extractor from a sentence and report it as JSON in the following format:
  Drugs: <List of extracted entities>
  Please report only Drugs. Do not report any other entities like Diseases, Genes, Proteins, or Enzymes."""

In [8]:
def drug_entity_extractor_v2(text):
    chat_model_id, chat_deployment_id = get_gpt35()
    prompt_updated = DRUG_ENTITY_EXTRACTION + "\n" + "Sentence : " + text
    resp = get_GPT_response(prompt_updated, DRUG_ENTITY_EXTRACTION, chat_model_id, chat_deployment_id, temperature=0)
    try:
        entity_dict = json.loads(resp)
        return entity_dict["Drugs"]
    except:
        return None

##  combined

In [9]:

def _safe_json_loads(text: str):
    """Strict JSON first; fallback to best-effort object between braces."""
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return None

def _norm(s):
    return s.strip().lower() if isinstance(s, str) else str(s).strip().lower()

def _dedup_norm_list(xs):
    seen, out = set(), []
    for x in xs or []:
        x2 = _norm(x)
        if x2 and x2 not in seen:
            seen.add(x2)
            out.append(x2)
    return out

In [10]:
# combined prompt (Drugs, Diseases, BiologicalProcesses)
ENTITY_EXTRACTION_PROMPT = """
You are an expert biomedical entity extractor. Extract entities from the input sentence and return strict JSON with the following schema:
{
  "Drugs": [<list of strings>],
  "Diseases": [<list of strings>],
  "BiologicalProcesses": [<list of strings>]
}
Rules:
- Return ONLY valid JSON (no code fences, no commentary).
- "Drugs": include small molecules and biotherapeutics (brand/generic names).
- "Diseases": include diseases and clinical syndromes (avoid genes/proteins).
- "BiologicalProcesses": include biological processes (e.g., 'apoptosis', 'temperature homeostasis', 'synaptic transmission'); prefer GO-style surface forms if present.
- Do NOT include genes, proteins, or enzymes.
- If none found for a category, return an empty list [] for that category.
- Preserve the surface forms present in the sentence.
"""


#extractor
def _extract_with_prompt(question_text: str, expected_keys=("Drugs","Diseases","BiologicalProcesses")):
    chat_model_id, chat_deployment_id = get_gpt35()
    prompt = ENTITY_EXTRACTION_PROMPT + "\nSentence: " + question_text
    raw = get_GPT_response(prompt, ENTITY_EXTRACTION_PROMPT, chat_model_id, chat_deployment_id, temperature=0)
    parsed = _safe_json_loads(raw) or {}
    return {k: (parsed.get(k, []) or []) for k in expected_keys}


# Metrics (micro P/R/F1; penalizes extra predictions as FP) 
def _metrics_multilabel(gold_single_list, pred_list_of_lists):
    """
    gold_single_list: list[str]   (exactly one gold per row)
    pred_list_of_lists: list[list[str]] (0..k predictions per row)
    """
    gold_sets = [[g] for g in gold_single_list]
    pred_sets = pred_list_of_lists

    mlb = MultiLabelBinarizer()
    mlb.fit(gold_sets + pred_sets)  

    Y_true = mlb.transform(gold_sets)
    Y_pred = mlb.transform(pred_sets)

    P = precision_score(Y_true, Y_pred, average='micro', zero_division=0)
    R = recall_score(Y_true, Y_pred, average='micro', zero_division=0)
    F = f1_score(Y_true, Y_pred, average='micro', zero_division=0)

    TP = int(((Y_true == 1) & (Y_pred == 1)).sum())
    FP = int(((Y_true == 0) & (Y_pred == 1)).sum())
    FN = int(((Y_true == 1) & (Y_pred == 0)).sum())
    N  = len(gold_single_list)
    hit_rate  = float(np.mean([g in p for g, p in zip(gold_single_list, pred_list_of_lists)]))
    avg_preds = float(np.mean([len(p) for p in pred_list_of_lists]))

    return {
        "N": N, "TP": TP, "FP": FP, "FN": FN,
        "precision": P, "recall": R, "f1": F,
        "hit_rate": hit_rate, "avg_preds_per_q": avg_preds
    }


# evaluator (works for Drug+Disease OR Disease+Process)
def evaluate_extraction(
    df: pd.DataFrame,
    text_col: str,
    gold_map: dict,
    n: int | None = None
):

    use_df = df.iloc[:n].copy() if isinstance(n, int) else df.copy()

    # Extract once per row
    preds_by_key = {k: [] for k in gold_map.keys()}
    for q in use_df[text_col]:
        try:
            out = _extract_with_prompt(q)
        except Exception:
            out = {}
        for k in gold_map.keys():
            preds_by_key[k].append(_dedup_norm_list(out.get(k, [])))

    # predictions, golds, hits
    for k, gold_colname in gold_map.items():
        pred_col = f"predicted_{k.lower()}"
        gold_col = f"{k.lower()}_gold"
        hit_col  = f"{k.lower()}_hit"

        use_df[pred_col] = preds_by_key[k]
        use_df[gold_col] = use_df[gold_colname].astype(str).map(_norm)
        use_df[hit_col]  = [g in p for g, p in zip(use_df[gold_col], use_df[pred_col])]

    # Per-target metrics
    metrics = {}
    for k in gold_map.keys():
        pred_col = f"predicted_{k.lower()}"
        gold_col = f"{k.lower()}_gold"
        metrics[k] = _metrics_multilabel(use_df[gold_col].tolist(), use_df[pred_col].tolist())

    # Joint accuracy across all targets
    hit_cols = [f"{k.lower()}_hit" for k in gold_map.keys()]
    use_df["all_hits"] = True
    for hc in hit_cols:
        use_df["all_hits"] &= use_df[hc]
    metrics["joint_accuracy"] = float(use_df["all_hits"].mean()) if len(use_df) else 0.0

    return use_df, metrics

# Gene Benchmark

In [11]:
sampled_questions = pd.read_csv("DMDB_benchmark/Benchmarks/DMDB_mechanistic_genes_filtered.csv")
sampled_questions.head(1)

,id,drug,Drug_MeshID,disease,protein,drug_name,disease_name,protein_name,protein_gene_symbol,question,count
0,['DB01219_MESH_C535694_1'],DB:DB01219,MESH:D003620,MESH:C535694,['UniProt:P21817'],Dantrolene,Malignant hyperthermia,['Ryanodine receptor 1'],['RYR1'],Which gene plays the most significant mechanistic role in how Drug Dantrolene treats or impacts the Disease Malignant hyperthermia?,1


In [12]:
sampled_questions.shape

(798, 11)

In [13]:
sampled_questions.columns

Index(['id', 'drug', 'Drug_MeshID', 'disease', 'protein', 'drug_name',
       'disease_name', 'protein_name', 'protein_gene_symbol', 'question',
       'count'],
      dtype='object')

In [14]:
eval_df_gg, metrics_gg = evaluate_extraction(
    df=sampled_questions,
    text_col="question",
    gold_map={"Drugs": "drug_name", "Diseases": "disease_name"},
    n=None
)
print("Drug metrics:", metrics_gg["Drugs"])
print("Disease metrics:", metrics_gg["Diseases"])
print("Joint accuracy:", metrics_gg["joint_accuracy"])


Drug metrics: {'N': 798, 'TP': 794, 'FP': 6, 'FN': 4, 'precision': 0.9925, 'recall': 0.9949874686716792, 'f1': 0.9937421777221527, 'hit_rate': 0.9949874686716792, 'avg_preds_per_q': 1.0025062656641603}
Disease metrics: {'N': 798, 'TP': 790, 'FP': 12, 'FN': 8, 'precision': 0.9850374064837906, 'recall': 0.9899749373433584, 'f1': 0.9875, 'hit_rate': 0.9899749373433584, 'avg_preds_per_q': 1.005012531328321}
Joint accuracy: 0.9849624060150376


In [15]:
eval_df_gg.head(5)[["question","drugs_gold","predicted_drugs","diseases_gold","predicted_diseases","drugs_hit","diseases_hit","all_hits"]]

,question,drugs_gold,predicted_drugs,diseases_gold,predicted_diseases,drugs_hit,diseases_hit,all_hits
0,Which gene plays the most significant mechanistic role in how Drug Dantrolene treats or impacts the Disease Malignant hyperthermia?,dantrolene,[dantrolene],malignant hyperthermia,[malignant hyperthermia],True,True,True
1,Which gene plays the most significant mechanistic role in how Drug Filgrastim treats or impacts the Disease Cyclical neutropenia?,filgrastim,[filgrastim],cyclical neutropenia,[cyclical neutropenia],True,True,True
2,Which gene plays the most significant mechanistic role in how Drug ezetimibe treats or impacts the Disease Sitosterolemia?,ezetimibe,[ezetimibe],sitosterolemia,[sitosterolemia],True,True,True
3,Which gene plays the most significant mechanistic role in how Drug Filgrastim treats or impacts the Disease Congenital neutropenia?,filgrastim,[filgrastim],congenital neutropenia,[congenital neutropenia],True,True,True
4,Which gene plays the most significant mechanistic role in how Drug cortisone acetate treats or impacts the Disease Humoral Hypercalcemia Of Malignancy?,cortisone acetate,[cortisone acetate],humoral hypercalcemia of malignancy,[humoral hypercalcemia of malignancy],True,True,True


In [16]:
eval_df_gg.shape

(798, 18)

# Metabolite

In [17]:
DMDB_chebi_metabolite_filtered_201qa = pd.read_csv("DMDB_benchmark/Benchmarks/DMDB_chebi_metabolite_filtered.csv")
DMDB_chebi_metabolite_filtered_201qa.shape

(201, 19)

In [18]:
DMDB_chebi_metabolite_filtered_201qa.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,metabolite,Drug_MeshID,node_names,drug_name,disease_name,metabolite_name,edges,question,metabolite_name_str
0,56,DB00994_MESH_D007634_1,DRUGBANK:DB00994,MESH:D007634,"['MESH:D009355', 'CHEBI:18111', 'GO:0006412', 'taxonomy:1280', 'MESH:D007634']",5,4,1,['Drug - ChemicalSubstance - BiologicalProcess - OrganismTaxon - Disease'],['Drug - decreases activity of - ChemicalSubstance - participates in - BiologicalProcess - in taxon - OrganismTaxon - causes - Disease'],CHEBI:18111,MESH:D009355,"['neomycin', 'ribosomal RNA', 'translation', 'Staphylococcus aureus', 'Keratitis']",neomycin,Keratitis,['ribosomal RNA'],"['decreases activity of', 'participates in', 'in taxon', 'causes']",Which biochemical entity is affected by the Drug neomycin via its mechanism of action in treating the Disease Keratitis ?,ribosomal RNA


In [19]:
eval_df_mb, metrics_mb = evaluate_extraction(
    df=DMDB_chebi_metabolite_filtered_201qa,
    text_col="question",
    gold_map={"Drugs": "drug_name", "Diseases": "disease_name"},
    n=None  
)
print("Drug metrics:", metrics_mb["Drugs"])
print("Disease metrics:", metrics_mb["Diseases"])
print("Joint accuracy:", metrics_mb["joint_accuracy"])


Drug metrics: {'N': 201, 'TP': 201, 'FP': 1, 'FN': 0, 'precision': 0.995049504950495, 'recall': 1.0, 'f1': 0.9975186104218362, 'hit_rate': 1.0, 'avg_preds_per_q': 1.0049751243781095}
Disease metrics: {'N': 201, 'TP': 198, 'FP': 5, 'FN': 3, 'precision': 0.9753694581280788, 'recall': 0.9850746268656716, 'f1': 0.9801980198019802, 'hit_rate': 0.9850746268656716, 'avg_preds_per_q': 1.0099502487562189}
Joint accuracy: 0.9850746268656716


In [20]:
eval_df_mb.head(5)[["question","drugs_gold","predicted_drugs","diseases_gold","predicted_diseases","drugs_hit","diseases_hit","all_hits"]]

,question,drugs_gold,predicted_drugs,diseases_gold,predicted_diseases,drugs_hit,diseases_hit,all_hits
0,Which biochemical entity is affected by the Drug neomycin via its mechanism of action in treating the Disease Keratitis ?,neomycin,[neomycin],keratitis,[keratitis],True,True,True
1,Which biochemical entity is affected by the Drug ivermectin via its mechanism of action in treating the Disease Infection by Onchocerca volvulus ?,ivermectin,[ivermectin],infection by onchocerca volvulus,[infection by onchocerca volvulus],True,True,True
2,Which biochemical entity is affected by the Drug formoterol via its mechanism of action in treating the Disease Pulmonary emphysema ?,formoterol,[formoterol],pulmonary emphysema,[pulmonary emphysema],True,True,True
3,"Which biochemical entity is affected by the Drug saxagliptin via its mechanism of action in treating the Disease Diabetes Mellitus, Type 2 ?",saxagliptin,[saxagliptin],"diabetes mellitus, type 2","[diabetes mellitus, type 2]",True,True,True
4,Which biochemical entity is affected by the Drug Capecitabine via its mechanism of action in treating the Disease Malignant tumor of stomach ?,capecitabine,[capecitabine],malignant tumor of stomach,[malignant tumor of stomach],True,True,True


In [21]:
eval_df_mb.shape

(201, 26)

# Drug-BP

In [22]:
DMDB_go_bp_df = pd.read_csv("DMDB_benchmark/Benchmarks/DMDB_go_bp_filtered.csv")
DMDB_go_bp_df.shape

(842, 16)

In [23]:
DMDB_go_bp_df.head(2)

,idx,id,drug,Drug_MeshID,disease,bp,drug_name,disease_name,bp_name,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,question
0,1,DB00619_MESH_D034721_1,DRUGBANK:DB00619,MESH:D000068877,MESH:D034721,GO:0008283,imatinib,Systemic mast cell disease,cell population proliferation,"['MESH:D000068877', 'UniProt:P10721', 'UniProt:P16234', 'GO:0008283', 'MESH:D034721']",5,5,1,['Drug - Protein - BiologicalProcess - Disease'],['Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease'],Which Drug can be used in the treatment of Systemic mast cell disease by targeting biological process: cell population proliferation?
1,3,DB00316_MESH_D005334_1,DRUGBANK:DB00316,MESH:D000082,MESH:D005334,GO:0001659,Acetaminophen,Fever,temperature homeostasis,"['MESH:D000082', 'reactome:R-HSA-2162123', 'UBERON:0000955', 'GO:0001659', 'MESH:D005334']",5,4,1,['Drug - Pathway - GrossAnatomicalStructure - BiologicalProcess - Disease'],['Drug - negatively regulates - Pathway - occurs in - GrossAnatomicalStructure - location of - BiologicalProcess - negatively correlated with - Disease'],Which Drug can be used in the treatment of Fever by targeting biological process: temperature homeostasis?


In [24]:
# Disease + Biological Process task
eval_df_bp, metrics_bp = evaluate_extraction(
    df=DMDB_go_bp_df,
    text_col="question",
    gold_map={"Diseases": "disease_name", "BiologicalProcesses": "bp_name"},
    n=None
)

print("Disease metrics:", metrics_bp["Diseases"])
print("Process metrics:", metrics_bp["BiologicalProcesses"])
print("Joint accuracy:", metrics_bp["joint_accuracy"])


Disease metrics: {'N': 842, 'TP': 796, 'FP': 78, 'FN': 46, 'precision': 0.9107551487414187, 'recall': 0.9453681710213777, 'f1': 0.9277389277389277, 'hit_rate': 0.9453681710213777, 'avg_preds_per_q': 1.0380047505938241}
Process metrics: {'N': 842, 'TP': 795, 'FP': 82, 'FN': 47, 'precision': 0.9064994298745724, 'recall': 0.9441805225653207, 'f1': 0.924956369982548, 'hit_rate': 0.9441805225653207, 'avg_preds_per_q': 1.0415676959619953}
Joint accuracy: 0.8919239904988123


In [25]:
eval_df_bp.head(5)[["question","diseases_gold","predicted_diseases","biologicalprocesses_gold","predicted_biologicalprocesses","diseases_hit","biologicalprocesses_hit","all_hits"]]

,question,diseases_gold,predicted_diseases,biologicalprocesses_gold,predicted_biologicalprocesses,diseases_hit,biologicalprocesses_hit,all_hits
0,Which Drug can be used in the treatment of Systemic mast cell disease by targeting biological process: cell population proliferation?,systemic mast cell disease,[systemic mast cell disease],cell population proliferation,[cell population proliferation],True,True,True
1,Which Drug can be used in the treatment of Fever by targeting biological process: temperature homeostasis?,fever,[fever],temperature homeostasis,[temperature homeostasis],True,True,True
2,Which Drug can be used in the treatment of Pain by targeting biological process: inflammatory response?,pain,[pain],inflammatory response,[inflammatory response],True,True,True
3,Which Drug can be used in the treatment of CMV infection by targeting biological process: viral DNA genome replication?,cmv infection,[cmv infection],viral dna genome replication,[viral dna genome replication],True,True,True
4,Which Drug can be used in the treatment of Bacterial septicemia by targeting biological process: bacterial Nucleic Acid synthesis?,bacterial septicemia,[bacterial septicemia],bacterial nucleic acid synthesis,[bacterial nucleic acid synthesis],True,True,True


In [26]:
eval_df_bp.shape

(842, 23)

# Summary

In [27]:
results_summary = pd.DataFrame([
    {
        "Benchmark": "Gene-centric: Drug-Gene-Disease",
        "N": metrics_gg["Drugs"]["N"],
        "Drug P/R/F1": f"{metrics_gg['Drugs']['precision']:.3f}/{metrics_gg['Drugs']['recall']:.3f}/{metrics_gg['Drugs']['f1']:.3f}",
        "Disease P/R/F1": f"{metrics_gg['Diseases']['precision']:.3f}/{metrics_gg['Diseases']['recall']:.3f}/{metrics_gg['Diseases']['f1']:.3f}",
        "Joint Accuracy": f"{metrics_gg['joint_accuracy']:.3f}"
    },
    {
        "Benchmark": "Metabolite-centric: Drug-Metabolite-Disease", 
        "N": metrics_mb["Drugs"]["N"],
        "Drug P/R/F1": f"{metrics_mb['Drugs']['precision']:.3f}/{metrics_mb['Drugs']['recall']:.3f}/{metrics_mb['Drugs']['f1']:.3f}",
        "Disease P/R/F1": f"{metrics_mb['Diseases']['precision']:.3f}/{metrics_mb['Diseases']['recall']:.3f}/{metrics_mb['Diseases']['f1']:.3f}",
        "Joint Accuracy": f"{metrics_mb['joint_accuracy']:.3f}"
    },
    {
        "Benchmark": "Drug-Bio.Process-centric: Drug-BiologicalProcess-Disease",
        "N": metrics_bp["Diseases"]["N"],
        "Disease P/R/F1": f"{metrics_bp['Diseases']['precision']:.3f}/{metrics_bp['Diseases']['recall']:.3f}/{metrics_bp['Diseases']['f1']:.3f}",
        "Bio-Process P/R/F1": f"{metrics_bp['BiologicalProcesses']['precision']:.3f}/{metrics_bp['BiologicalProcesses']['recall']:.3f}/{metrics_bp['BiologicalProcesses']['f1']:.3f}",
        "Joint Accuracy": f"{metrics_bp['joint_accuracy']:.3f}"
    }
])

In [28]:
results_summary

,Benchmark,N,Drug P/R/F1,Disease P/R/F1,Joint Accuracy,Bio-Process P/R/F1
0,Gene-centric: Drug-Gene-Disease,798,0.993/0.995/0.994,0.985/0.990/0.988,0.985,NaN
1,Metabolite-centric: Drug-Metabolite-Disease,201,0.995/1.000/0.998,0.975/0.985/0.980,0.985,NaN
2,Drug-Bio.Process-centric: Drug-BiologicalProcess-Disease,842,NaN,0.911/0.945/0.928,0.892,0.906/0.944/0.925
